# To Replicate Our Results

Run all cells.

**NOTE**: the files in `./raw_data` needed to be unzipped for this to run. 

In [ ]:
import os  
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from scipy import stats
from scipy.signal import resample

In [ ]:
# NOTE: We modified this cell to link to our local paths.
PATH = "./raw_data/"
SAVE_PATH = "./segmented_data/"

In [ ]:
def GetLeftRightCode(path):
    _lrcode = []
    lrcode = []
    for line in open(path,'r'):
        _lrcode.append(line)
    for i in range(len(_lrcode)):
        lrcode.append(_lrcode[i][7:-2])
    return lrcode
#print GetLeftRightCode("D:\Datasets\MotionData\MotionDataB\wb_pos_info.txt")

def SegmentData(motion_data, seg_size, action, lrcode = True):
    segments = np.empty((0,seg_size,3))
    labels = np.empty((0))
    l_ing_data_x,l_ing_data_y,l_ing_data_z = [],[],[]   
    r_ing_data_x,r_ing_data_y,r_ing_data_z = [],[],[]  
    for i in range(len(motion_data)):
        if motion_data[i][6] == action:
            if motion_data[i][2] == lrcode[0]: 
                l_ing_data_x.append(motion_data[i][3])
                l_ing_data_y.append(motion_data[i][4])
                l_ing_data_z.append(motion_data[i][5])
            elif motion_data[i][2] == lrcode[1]:
                r_ing_data_x.append(motion_data[i][3])
                r_ing_data_y.append(motion_data[i][4])
                r_ing_data_z.append(motion_data[i][5])

    _start = 0
    _end = int(seg_size/2)
    
    print(type(_start), type(_end))
    
    for j in range(len(l_ing_data_x)):   

        _end = int(_end + seg_size/2)
        if _end <= len(l_ing_data_x):
            _x = l_ing_data_x[_start:_end]
            _y = l_ing_data_y[_start:_end]
            _z = l_ing_data_z[_start:_end]
            segments = np.vstack([segments, np.dstack([_x,_y,_z])])#3xseg_size
            labels = np.append(labels, action)
            _start = int(_start + seg_size/2)
        else:
            break
    
    _start = 0
    _end = int(seg_size/2)
    
    for j in range(len(r_ing_data_x)):   

        _end = int(_end + seg_size/2)
        if _end <= len(r_ing_data_x):
            _x = r_ing_data_x[_start:_end]
            _y = r_ing_data_y[_start:_end]
            _z = r_ing_data_z[_start:_end]
            segments = np.vstack([segments, np.dstack([_x,_y,_z])])#3xseg_size
            labels = np.append(labels, action)
            _start = int(_start + seg_size/2)
        else:
            break
    
    return segments, labels

def AverageFilter(l, windowsize): #list windowsize default is 3,if change needs to change weights and N
    _l = []
    N = (windowsize-1)/2
    for i in range(len(l)):
        if i >= N and i <= len(l)-1-N:
            #print(N, i, i-N, i+N+1)
            _l.append(np.average(l[int(i-N):int(i+N+1)],weights=[1 for j in range(windowsize)]))
        else:
            _l.append(l[int(i)])
    return _l


In [ ]:
# NOTE: we added these helper functions to put the PAAWS data in an easily parsable format compatible with this code. 

import pandas as pd
from typing import Tuple
from datetime import datetime, timedelta

def read_data(file: str, agd: bool = False, num_rows = len_ds_10) -> Tuple[datetime, pd.DataFrame]:
    """
    Reads the actigraph data file and returns the starting timestamp and the
    corresponding DataFrame.

    Parameters
    ----------
    file : string
        Path to the actigraph file (e.g., accel, IMU, or HR data).

    agd : bool
        If True, assume a 1-second interval for sampling.

    Returns
    ----------
    start : timedelta
        The starting timestamp.

    df : pd.Dataframe
        The actigraph data as a pd.DataFrame.
    """

    sampling_rate = 1
    start_date = None
    start_time = None

    # Open the file and read metadata.
    with open(file) as f:
        line = f.readline()
        parsed = line.split()

        for i in range(len(parsed)):
            if parsed[i] == "Hz":
                sampling_rate = int(parsed[i - 1])  # Get the sampling rate.
                break

        f.readline()
        start_time = f.readline().split()[-1]  # Get start time.
        start_date = f.readline().split()[-1]  # Get start date.

    start = datetime.strptime(start_date + " " + start_time, "%m/%d/%Y %H:%M:%S")

    # Calculate the time step between each sample.
    step = timedelta(seconds=1 / sampling_rate)
    if agd:
        step = timedelta(seconds=1)  # Use 1 second for AGD format.

    # Read accel data into a DataFrame (skip first 10 rows of metadata).
    df = pd.read_csv(file, skiprows=10, header=0, nrows=num_rows)

    print(df.shape, num_rows)

    # Add timestamps for each data point to the dataframe
    df["Timestamp"] = [start + i * step for i in range(len(df))]
    
    return start, df


def add_label_to_actigraph(actigraph, label) -> pd.DataFrame:
    """
    Adds activity labels to the actigraph data based on the time intervals
    in the label data.

    Parameters
    ----------
    actigraph : pd.DataFrame
        DataFrame containing actigraph data.

    label : pd.DataFrame
        DataFrame containing labeled activity data with start and stop times.

    Returns
    ----------
    actigraph : pd.DataFrame
        DataFrame with added 'Activity' column containing the activity class
        from the labeled data.
    """

    actigraph["Activity"] = None

    # Denote data before and after data collection.
    data_start = label["START_TIME"].iloc[0]#, "Activity"
    data_end = label["STOP_TIME"].iloc[-1]#, "Activity"
    before_string = "Before_Data_Collection"
    after_string = "After_Data_Collection"

    print(actigraph)
    print(data_start, data_end, before_string, after_string)

    print("ACC WHOLE", actigraph.loc[actigraph["Timestamp"] < data_start])
    print("ACT", actigraph.loc[actigraph["Timestamp"] < data_start, "Activity"])
    actigraph.loc[actigraph["Timestamp"] < data_start, "Activity"] = before_string
    print("before")
    actigraph.loc[actigraph["Timestamp"] > data_end, "Activity"] = after_string
    ("after")

    # Assign the activity label.
    for _, row in label.iterrows():
        start = row["START_TIME"]
        stop = row["STOP_TIME"]
        actigraph.loc[
            (actigraph["Timestamp"] >= start) & (actigraph["Timestamp"] <= stop),
            "Activity",
        ] = row["ACTIVITY_CLASS"]

    print("all")
    return actigraph


def data_to_csv(actigraph_path: str, label_path: str) -> None:
    """
    Combines actigraph data with activity labels and saves the result as a CSV.

    Parameters
    ----------
    actigraph_path : string
        Path to the input actigraph file.

    label_path : string
        Path to the input label file containing activity intervals.

    output_path : string
        Path where the combined data should be saved as a CSV file.

    Returns
    ----------
    None. Actigraph data with timestamp and labels is saved to the specified
    output path.
    """

    # Read actigraph data.
    print("get csv")
    _, actigraph = read_data(actigraph_path)

    # Read label data and map the activity types to the activity classes.
    label = pd.read_csv(label_path, parse_dates=["START_TIME", "STOP_TIME"])
    print("get lab")
    mapping = {l : l for l in label["PA_TYPE"]}
    print(mapping)
    #mapping = MAPPING_SCHEMES["lab_fl_5"]  #  Default to 5 activity classes.
    label["ACTIVITY_CLASS"] = [mapping.get(x, None) for x in label["PA_TYPE"]]

    print(actigraph.shape)

    actigraph = add_label_to_actigraph(actigraph, label)

    # Save the merged data to a CSV file.
    
    #actigraph.to_csv(output_path, index=False)
    return actigraph

# Helper code from Claude (Sonnet 4.1) to resample labels. 
def resample_labels_nearest(labels, original_fs=100, target_fs=52):
    original_duration = len(labels) / original_fs
    n_samples_new = int(original_duration * target_fs)
    
    t_original = np.linspace(0, original_duration, len(labels), endpoint=False)
    t_target = np.linspace(0, original_duration, n_samples_new, endpoint=False)
    
    indices = np.searchsorted(t_original, t_target, side='left')
    
    # Handle edge case where index equals length
    indices = np.clip(indices, 0, len(labels) - 1)
    
    # Use nearest neighbor logic
    for i, target_time in enumerate(t_target):
        left_idx = max(0, indices[i] - 1) if indices[i] > 0 else 0
        right_idx = min(len(labels) - 1, indices[i])
        
        if left_idx == right_idx:
            indices[i] = left_idx
        else:
            # Choose the temporally closer sample
            left_dist = abs(target_time - t_original[left_idx])
            right_dist = abs(target_time - t_original[right_idx])
            indices[i] = left_idx if left_dist <= right_dist else right_idx
    
    return np.array(labels)[indices]

In [ ]:
# NOTE: We modified this code to use NumPy for slightly faster performance and to work with data
# from only one wrist. The functionality remains the same. 

def AveragedSegmentData(motion_data, seg_size, action, lrcode = True):
    segments = np.empty((0, seg_size, 3))

    labels = motion_data[:, 3:] # timestamp and label
    labels_final = []

    motion_data = motion_data[:, :3]
    motion_data = np.array(motion_data[:, :], dtype=float)

    l_ing_data_x = motion_data[:, 0]
    l_ing_data_y = motion_data[:, 1]
    l_ing_data_z = motion_data[:, 2]

    al_ing_data_x = AverageFilter(l_ing_data_x,11)
    al_ing_data_y = AverageFilter(l_ing_data_y,11)
    al_ing_data_z = AverageFilter(l_ing_data_z,11)
    _start = 0
    _end = seg_size
    for j in range(len(l_ing_data_x)):   
        if _end <= len(l_ing_data_x):
            
            _x = al_ing_data_x[_start:_end]
            _y = al_ing_data_y[_start:_end]
            _z = al_ing_data_z[_start:_end]
            labels_final.append(labels[_start]) # TODO: added to make the lables the same size.

            new_row = np.array([_x, _y, _z])
            new_row = np.reshape(new_row, (-1, 256, 3))
            segments = np.vstack([segments, new_row])
            _start = int(_start + seg_size)
            _end = int(_end + seg_size)

        else:
            break
    
    return segments, labels_final

In [ ]:
# NOTE: We modified this cell to work with our data. 

sides = ["LeftWrist", "RightWrist"]

seg_size = 256

for side in sides:
    side_path = PATH+f"DS_10-Free-{side}.csv"
    lab_path = PATH+"DS_10-Free-label.csv"
    
    X = np.empty((0, seg_size, 3))
    Y = np.empty((2)) # NOTE: Modified to save labels and timestamp. 
    
    data = data_to_csv(side_path, lab_path)
    data.to_csv("./CHECK_DATA_IS_OKAY.csv", index=False)

    motion_data = []# 0 time w_id x y z activies
    print(data.iloc[0])

            # TODO: I need to downsample here! 
            # REMOVE THESE TWO VP
            # motion_data = []
            # data = pd.read_csv("./CHECK_DATA_IS_OKAY.csv", low_memory=False, nrows=2000)

    accel = data[["Accelerometer X", "Accelerometer Y", "Accelerometer Z"]].to_numpy()
    labels = data[["Timestamp", "Activity"]].to_numpy()
    print("pre acc - lab", accel.shape, labels.shape)
            
    lab_2 = resample_labels_nearest(labels)
    print("lab2", lab_2.shape)
            
    acc = resample(accel, lab_2.shape[0], axis=0) # NOTE: downsampled data to match the Hz of ASTRI/ADL.
            #print(acc)
    print("acc", acc.shape)
            
    data = np.concatenate([acc, lab_2], axis=1)
    print(data.shape, data[0])
    data = pd.DataFrame(data)
    for i in range(data.shape[0]):
        #print(data.iloc[i])
        motion_data.append(list(map(lambda x : x, data.iloc[i])))
                #print(list(map(lambda x : x, data.iloc[i])))
        print("MOTION DATA 0", motion_data[0])

    motion_data = np.array(motion_data)
    # print("TEST LABS", motion_data[:, 3:])
    #         #motion_data = motion_data[:, 3:]
    #         print(motion_data[:, -1])
    #         print(motion_data.shape)
            #pd.DataFrame(motion_data, columns=["Accelerometer X", "Accelerometer Y", "Accelerometer Z", "Timestamp", "Activity"]).to_csv("./CHECK_MOTION_DATA.csv")
            
    for action in ["TEST"]:
        print(action)
        segments, labels = AveragedSegmentData(motion_data, seg_size, action[1:], "blank")
        print(len(labels))
        Y = np.array(labels)
                # X = np.append(X, segments)
                # Y = np.append(Y, labels)#label walking

        print(X.shape)
        print(segments.shape)
                #print(X[0], X[1], X[2])
                #print(segments[0])
        X = np.vstack((X, segments))
                #print(X[0])
                #pd.DataFrame(X, columns=["x", "y", "z",]).to_csv("./FINAL_CHECK_MOTION_DATA.csv")
                #print(segments.shape, X.shape)
        print(Y.shape, "LAB_SHAPE")#, "Resamp?", labels[128::128].shape, labels[128::128])

        # Save the segmented data and the labels. 
        np.save(SAVE_PATH+f'{side}.npy', X)
        pd.DataFrame(labels).to_csv(SAVE_PATH+f'{side}_labels.csv')

/projects/annemarie/PAAWS_backup/for_release/PAAWS_FreeLiving/DS_10/label/DS_10-Free-label.csv
/projects/annemarie/PAAWS_backup/for_release/PAAWS_FreeLiving/DS_10/accel/
['DS_10-Free-LeftWrist.csv', 'DS_10-Free-RightAnkle.csv', 'DS_10-Free-RightThigh.csv', 'DS_10-Free-RightWaist.csv', 'DS_10-Free-RightWrist.csv']
CSV path /projects/annemarie/PAAWS_backup/for_release/PAAWS_FreeLiving/DS_10/accel/DS_10-Free-LeftWrist.csv
get csv
(8063921, 3) 8063921
get lab
{'PA_Type_Video_Unavailable/Indecipherable': 'PA_Type_Video_Unavailable/Indecipherable', 'Standing_With_Movement': 'Standing_With_Movement', 'Sitting_With_Movement': 'Sitting_With_Movement', 'Puttering_Around': 'Puttering_Around', 'Walking': 'Walking', 'Walking_Fast': 'Walking_Fast', 'Walking_Down_Stairs': 'Walking_Down_Stairs', 'Walking_Up_Stairs': 'Walking_Up_Stairs', 'Washing_Hands': 'Washing_Hands', 'Synchronizing_Sensors': 'Synchronizing_Sensors', 'Folding_Clothes': 'Folding_Clothes', 'PA_Type_Other': 'PA_Type_Other', 'Kneeling_W